In [1]:
import torch
from torch import nn as nn
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator
from mpl_toolkits.mplot3d import axes3d
import scipy.io
from torch.utils.data import DataLoader, random_split, TensorDataset
import pandas as pd
import time

from utils import GiveMeDataMuscleAbove, GiveMeDataMuscleBelow, CreateTensorLoader, plotplot, train_loop, test_loop, plotLossEpoch, plotSemilogyLossEpoch, train_loop_FA, test_loop_FA
%matplotlib inline
%matplotlib widget

In [2]:
torch.manual_seed(66)
torch.cuda.manual_seed(66)
torch.backends.cudnn.deterministic = True
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.net=nn.Sequential(
            nn.Linear(4,128,dtype=torch.float32),
            nn.Softplus(),
            nn.Linear(128,128,dtype=torch.float32),
            nn.Softplus(),
            nn.Linear(128,1,dtype=torch.float32),
            )
    def forward(self, x):
        return self.net(x)

In [3]:
file_path = r"C:\\Users\\gwang\\OneDrive - INSA Lyon\\code Python\\FastAdaptation\\code\\data_blm\\sinus_biglongmuscle_0g.mat"
# Load the .mat file
mat = scipy.io.loadmat(file_path)
data_train = mat['data']

train_ratio = .8
batch_size = 1024
seed = 66
gpu = 1
e_h, deh_dt, P_h, dPdt_h, q_h = GiveMeDataMuscleAbove(data_train)
train_loader, test_loader = CreateTensorLoader(e_h, deh_dt, P_h, dPdt_h, q_h, 10, train_ratio, batch_size, seed, gpu)
# plotplot(data_train)

Data moved to GPU sucessfully.


In [4]:
net_blm = Net()
net_blm.to('cuda') if gpu else net_blm.to('cpu')

Net(
  (net): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): Softplus(beta=1, threshold=20)
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Softplus(beta=1, threshold=20)
    (4): Linear(in_features=128, out_features=1, bias=True)
  )
)

In [5]:
loss_function = nn.MSELoss()
optim_blm = torch.optim.Adam(Net.parameters(net_blm), lr=1e-4, weight_decay=1e-5)

In [6]:
Loss_train_blm = []
Loss_test_blm =[]

In [7]:
print("Model and data is currently on:", next(net_blm.parameters()).device)
print(f"Optimizer = {type(optim_blm).__name__}\nLoss Function = {type(loss_function).__name__}")

target_loss=.001
start_time = time.time()
for t in range(10000):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss = train_loop(train_loader, net_blm, loss_function, optim_blm, Loss_train_blm, batch_size)
    test_loss = test_loop(test_loader, net_blm, loss_function)
    Loss_test_blm.append(test_loss)
    if test_loss <= target_loss and train_loss <= target_loss:
        print(test_loss)
        break

end_time = time.time()
execution_time = end_time - start_time
execution_time

Model and data is currently on: cuda:0
Optimizer = Adam
Loss Function = MSELoss
Epoch 1
-------------------------------
loss: 282.179352  [ 1024/30952]
loss: 328.534821  [11264/30952]
loss: 296.159119  [21504/30952]
loss: 464.817413  [30952/30952]
Avg training loss: 321.200688
Avg Testing loss: 285.255363 

Epoch 2
-------------------------------
loss: 378.370300  [ 1024/30952]
loss: 377.868835  [11264/30952]
loss: 249.391495  [21504/30952]
loss: 266.799957  [30952/30952]
Avg training loss: 321.200322
Avg Testing loss: 285.254997 

Epoch 3
-------------------------------
loss: 294.509918  [ 1024/30952]
loss: 295.614807  [11264/30952]
loss: 356.055969  [21504/30952]
loss: 355.525604  [30952/30952]
Avg training loss: 321.199853
Avg Testing loss: 285.254519 

Epoch 4
-------------------------------
loss: 235.490051  [ 1024/30952]
loss: 304.430908  [11264/30952]
loss: 274.100769  [21504/30952]
loss: 322.965393  [30952/30952]
Avg training loss: 321.199200
Avg Testing loss: 285.253847 

Epoc

In [ ]:
plotLossEpoch(Loss_train_blm, Loss_test_blm,0, 'loss_blm.svg')
plotSemilogyLossEpoch(Loss_train_blm, Loss_test_blm, 0, 'semilogy_loss_blm.svg')

In [ ]:
torch.save(net_blm.state_dict(), f"net_blm.pth")
np.save(f"loss_blm_train.npy", Loss_train_blm)
np.save(f"loss_blm_test.npy", Loss_test_blm)